In [1]:
# ==============================================================================
# NB03_Candidate_Generation
# PHASE 4 – CANDIDATE GENERATION
#
# Input:
#   Silver_Products (Fabric Lakehouse Delta table)
#
# Output:
#   ProductCandidatePairs (Fabric Lakehouse Delta table)
#
# Expected source counts:
#   StyleHaus   = 94
#   UrbanThread = 70
#
# Expected candidate output:
#   185 rows, 54 columns
#
# NOTE:
#   This notebook generates and validates candidate pairs. It does NOT decide
#   whether two products are duplicates
# ==============================================================================

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 3, Finished, Available, Finished, False)

In [2]:
# ==============================================================================
# CELL 1 – IMPORTS AND CONFIGURATION
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

SOURCE_TABLE = "Silver.Products"
OUTPUT_TABLE = "semantic.ProductCandidatePairs"

STYLEHAUS_SOURCE = "StyleHaus"
URBANTHREAD_SOURCE = "UrbanThread"

EXPECTED_STYLEHAUS_COUNT = 94
EXPECTED_URBANTHREAD_COUNT = 70
EXPECTED_CANDIDATE_COUNT = 185
EXPECTED_OUTPUT_COLUMNS = 54

SCHEMA_VERSION = "1.0"
SIMILARITY_TOLERANCE = 1e-8

print("Phase 4 – Candidate Generation")
print("Input table:", SOURCE_TABLE)
print("Output table:", OUTPUT_TABLE)

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 4, Finished, Available, Finished, False)

Phase 4 – Candidate Generation
Input table: Silver.Products
Output table: semantic.ProductCandidatePairs


In [3]:
# ==============================================================================
# CELL 2 – LOAD AND VALIDATE SILVER SOURCE
# ==============================================================================

silver_df = spark.read.table(SOURCE_TABLE)

required_columns = [
    "SourceSystem",
    "SourceProductID",
    "ProductName",
    "Brand",
    "Category",
    "Description",
    "Price",
    "Color",
    "Size",
    "Material",
    "InStock",
    "UpdatedAt",
    "BatchID",
    "IngestionTimestamp"
]

missing_columns = [
    c for c in required_columns
    if c not in silver_df.columns
]

assert not missing_columns, (
    f"Silver_Products is missing required columns: {missing_columns}"
)

silver_df = silver_df.select(*required_columns)

source_counts = {
    row["SourceSystem"]: row["count"]
    for row in silver_df.groupBy("SourceSystem").count().collect()
}

stylehaus_count = source_counts.get(STYLEHAUS_SOURCE, 0)
urbanthread_count = source_counts.get(URBANTHREAD_SOURCE, 0)

print("Total Silver records:", silver_df.count())
print("StyleHaus records:", stylehaus_count)
print("UrbanThread records:", urbanthread_count)

assert stylehaus_count == EXPECTED_STYLEHAUS_COUNT
assert urbanthread_count == EXPECTED_URBANTHREAD_COUNT

# Product IDs must be unique within each source.
duplicate_source_ids = (
    silver_df
    .groupBy("SourceSystem", "SourceProductID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_source_ids == 0, (
    f"Duplicate source product IDs found: {duplicate_source_ids}"
)

print("PASS: Silver source data loaded and checked.")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 5, Finished, Available, Finished, False)

Total Silver records: 164
StyleHaus records: 94
UrbanThread records: 70
PASS: Silver source data loaded and checked.


In [4]:
# ==============================================================================
# CELL 3 – DEFINE NORMALIZATION FUNCTIONS
# ==============================================================================

def normalize_text(column_name):
    """
    Lowercase, remove apostrophes, replace punctuation with spaces,
    collapse repeated whitespace, and trim.
    """
    value = F.lower(
        F.coalesce(F.col(column_name).cast("string"), F.lit(""))
    )

    value = F.regexp_replace(value, r"'", "")
    value = F.regexp_replace(value, r"[^a-z0-9]+", " ")
    value = F.regexp_replace(value, r"\s+", " ")

    return F.trim(value)


def normalize_color(column_name):
    """
    Normalize color text and common abbreviations.
    Examples: WHT -> white, BLK -> black, GRY -> grey.
    """
    value = normalize_text(column_name)

    color_aliases = {
        "blk": "black",
        "wht": "white",
        "gry": "grey",
        "grn": "green",
        "blu": "blue",
        "brn": "brown",
        "nvy": "navy"
    }

    for abbreviation, full_value in color_aliases.items():
        value = F.regexp_replace(
            value,
            rf"\b{abbreviation}\b",
            full_value
        )

    return F.trim(F.regexp_replace(value, r"\s+", " "))


def normalize_size(column_name):
    """
    Preserve sizing systems. Do not remove US/UK/EU prefixes.
    Examples: US 10 -> us 10; UK 10 -> uk 10; 32x30 -> 32x30.
    """
    value = F.lower(
        F.coalesce(F.col(column_name).cast("string"), F.lit(""))
    )
    return F.trim(F.regexp_replace(value, r"\s+", " "))


def jaccard_similarity(column1, column2):
    """
    Jaccard similarity over unique whitespace-delimited tokens.
    Returns a value between 0 and 1.
    """
    tokens1 = F.split(
        F.trim(F.coalesce(F.col(column1), F.lit(""))),
        r"\s+"
    )
    tokens2 = F.split(
        F.trim(F.coalesce(F.col(column2), F.lit(""))),
        r"\s+"
    )

    tokens1 = F.array_distinct(
        F.filter(tokens1, lambda x: x != "")
    )
    tokens2 = F.array_distinct(
        F.filter(tokens2, lambda x: x != "")
    )

    intersection_count = F.size(
        F.array_intersect(tokens1, tokens2)
    )
    union_count = F.size(
        F.array_union(tokens1, tokens2)
    )

    return (
        F.when(union_count == 0, F.lit(0.0))
         .otherwise(
             intersection_count.cast("double") /
             union_count.cast("double")
         )
    )


print("Normalization and similarity functions defined.")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 6, Finished, Available, Finished, False)

Normalization and similarity functions defined.


In [5]:
# ==============================================================================
# CELL 4 – NORMALIZE MATCHING ATTRIBUTES
# ==============================================================================

candidate_base_df = (
    silver_df
    .withColumn("NormalizedBrand", normalize_text("Brand"))
    .withColumn("NormalizedProductName", normalize_text("ProductName"))
    .withColumn("NormalizedCategory", normalize_text("Category"))
    .withColumn("NormalizedColor", normalize_color("Color"))
    # Keep US / UK / EU size systems intact.
    .withColumn("NormalizedSize", normalize_size("Size"))
    .withColumn("NormalizedMaterial", normalize_text("Material"))
    .withColumn("NormalizedDescription", normalize_text("Description"))
)

# Canonical brand aliases.
# Add aliases here only when two source brand labels represent the same brand.
canonical_brand_aliases = {
    "hoka one one": "hoka",
    "levi s": "levis"
}

alias_expression = F.create_map(
    *[
        item
        for pair in canonical_brand_aliases.items()
        for item in (F.lit(pair[0]), F.lit(pair[1]))
    ]
)

candidate_base_df = (
    candidate_base_df
    .withColumn(
        "CanonicalBrand",
        F.coalesce(
            F.element_at(alias_expression, F.col("NormalizedBrand")),
            F.col("NormalizedBrand")
        )
    )
)

normalized_columns = [
    "NormalizedBrand",
    "NormalizedProductName",
    "NormalizedCategory",
    "NormalizedColor",
    "NormalizedSize",
    "NormalizedMaterial",
    "NormalizedDescription",
    "CanonicalBrand"
]

normalization_invalid_counts = {
    column: candidate_base_df.filter(
        F.col(column).isNull() | (F.trim(F.col(column)) == "")
    ).count()
    for column in normalized_columns
}

print("Normalized source records:", candidate_base_df.count())
for column, invalid_count in normalization_invalid_counts.items():
    print(f"{column}: {invalid_count} invalid records")

assert all(v == 0 for v in normalization_invalid_counts.values()), (
    f"Invalid normalized values found: {normalization_invalid_counts}"
)

# Confirm normalization did not alter original source attributes.
original_columns = required_columns

original_difference_count = (
    silver_df.select(*original_columns)
    .exceptAll(candidate_base_df.select(*original_columns))
    .count()
)

assert original_difference_count == 0
print("PASS: Original Silver attributes are preserved.")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 7, Finished, Available, Finished, False)

Normalized source records: 164
NormalizedBrand: 0 invalid records
NormalizedProductName: 0 invalid records
NormalizedCategory: 0 invalid records
NormalizedColor: 0 invalid records
NormalizedSize: 0 invalid records
NormalizedMaterial: 0 invalid records
NormalizedDescription: 0 invalid records
CanonicalBrand: 0 invalid records
PASS: Original Silver attributes are preserved.


In [6]:
# ==============================================================================
# CELL 5 – PREPARE SOURCE-SPECIFIC DATAFRAMES
# ==============================================================================

stylehaus_df = (
    candidate_base_df
    .filter(F.col("SourceSystem") == STYLEHAUS_SOURCE)
)

urbanthread_df = (
    candidate_base_df
    .filter(F.col("SourceSystem") == URBANTHREAD_SOURCE)
)

print("StyleHaus records:", stylehaus_df.count())
print("UrbanThread records:", urbanthread_df.count())

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 8, Finished, Available, Finished, False)

StyleHaus records: 94
UrbanThread records: 70


In [7]:
# ==============================================================================
# CELL 6 – GENERATE CROSS-SOURCE CANDIDATE PAIRS
# ==============================================================================
# Blocking strategy:
#   StyleHaus.CanonicalBrand = UrbanThread.CanonicalBrand
#
# This is a candidate-generation rule, not a final duplicate decision.
# Size, color, category, and material are NOT mandatory blocking conditions.
# This preserves possible variants and avoids prematurely excluding matches.
# ==============================================================================

sh = stylehaus_df.alias("SH")
ut = urbanthread_df.alias("UT")

candidate_pairs_raw_df = (
    sh.join(
        ut,
        F.col("SH.CanonicalBrand") == F.col("UT.CanonicalBrand"),
        "inner"
    )
    .select(
        F.col("SH.SourceProductID").alias("StyleHaus_ProductID"),
        F.col("UT.SourceProductID").alias("UrbanThread_ProductID"),

        F.col("SH.BatchID").alias("StyleHaus_BatchID"),
        F.col("UT.BatchID").alias("UrbanThread_BatchID"),

        F.col("SH.IngestionTimestamp").alias(
            "StyleHaus_IngestionTimestamp"
        ),
        F.col("UT.IngestionTimestamp").alias(
            "UrbanThread_IngestionTimestamp"
        ),

        F.col("SH.ProductName").alias("StyleHaus_ProductName"),
        F.col("UT.ProductName").alias("UrbanThread_ProductName"),

        F.col("SH.Brand").alias("StyleHaus_Brand"),
        F.col("UT.Brand").alias("UrbanThread_Brand"),

        F.col("SH.Category").alias("StyleHaus_Category"),
        F.col("UT.Category").alias("UrbanThread_Category"),

        F.col("SH.Description").alias("StyleHaus_Description"),
        F.col("UT.Description").alias("UrbanThread_Description"),

        F.col("SH.Price").alias("StyleHaus_Price"),
        F.col("UT.Price").alias("UrbanThread_Price"),

        F.col("SH.Color").alias("StyleHaus_Color"),
        F.col("UT.Color").alias("UrbanThread_Color"),

        F.col("SH.Size").alias("StyleHaus_Size"),
        F.col("UT.Size").alias("UrbanThread_Size"),

        F.col("SH.Material").alias("StyleHaus_Material"),
        F.col("UT.Material").alias("UrbanThread_Material"),

        F.col("SH.InStock").alias("StyleHaus_InStock"),
        F.col("UT.InStock").alias("UrbanThread_InStock"),

        F.col("SH.UpdatedAt").alias("StyleHaus_UpdatedAt"),
        F.col("UT.UpdatedAt").alias("UrbanThread_UpdatedAt"),

        F.col("SH.CanonicalBrand").alias("CanonicalBrand"),

        F.col("SH.NormalizedProductName").alias(
            "StyleHaus_NormalizedProductName"
        ),
        F.col("UT.NormalizedProductName").alias(
            "UrbanThread_NormalizedProductName"
        ),

        F.col("SH.NormalizedCategory").alias(
            "StyleHaus_NormalizedCategory"
        ),
        F.col("UT.NormalizedCategory").alias(
            "UrbanThread_NormalizedCategory"
        ),

        F.col("SH.NormalizedColor").alias(
            "StyleHaus_NormalizedColor"
        ),
        F.col("UT.NormalizedColor").alias(
            "UrbanThread_NormalizedColor"
        ),

        F.col("SH.NormalizedSize").alias(
            "StyleHaus_NormalizedSize"
        ),
        F.col("UT.NormalizedSize").alias(
            "UrbanThread_NormalizedSize"
        ),

        F.col("SH.NormalizedMaterial").alias(
            "StyleHaus_NormalizedMaterial"
        ),
        F.col("UT.NormalizedMaterial").alias(
            "UrbanThread_NormalizedMaterial"
        ),

        F.col("SH.NormalizedDescription").alias(
            "StyleHaus_NormalizedDescription"
        ),
        F.col("UT.NormalizedDescription").alias(
            "UrbanThread_NormalizedDescription"
        )
    )
)

print("Generated candidate pairs:", candidate_pairs_raw_df.count())

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 9, Finished, Available, Finished, False)

Generated candidate pairs: 185


In [8]:
# ==============================================================================
# CELL 7 – CALCULATE CANDIDATE PAIR ID AND MATCHING SIGNALS
# ==============================================================================

candidate_pairs_with_signals_df = (
    candidate_pairs_raw_df

    .withColumn(
        "CandidatePairID",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("StyleHaus_ProductID"),
                F.col("UrbanThread_ProductID")
            ),
            256
        )
    )

    .withColumn(
        "CandidateSchemaVersion",
        F.lit(SCHEMA_VERSION)
    )

    .withColumn(
        "CanonicalBrandMatch",
        F.lit(True)
    )

    .withColumn(
        "OriginalBrandExactMatch",
        F.lower(F.trim(F.col("StyleHaus_Brand"))) ==
        F.lower(F.trim(F.col("UrbanThread_Brand")))
    )

    .withColumn(
        "ProductNameExactMatch",
        F.col("StyleHaus_NormalizedProductName") ==
        F.col("UrbanThread_NormalizedProductName")
    )

    .withColumn(
        "ProductNameSimilarity",
        jaccard_similarity(
            "StyleHaus_NormalizedProductName",
            "UrbanThread_NormalizedProductName"
        )
    )

    .withColumn(
        "CategoryMatch",
        F.col("StyleHaus_NormalizedCategory") ==
        F.col("UrbanThread_NormalizedCategory")
    )

    .withColumn(
        "ColorMatch",
        F.col("StyleHaus_NormalizedColor") ==
        F.col("UrbanThread_NormalizedColor")
    )

    .withColumn(
        "SizeMatch",
        F.col("StyleHaus_NormalizedSize") ==
        F.col("UrbanThread_NormalizedSize")
    )

    .withColumn(
        "MaterialMatch",
        F.col("StyleHaus_NormalizedMaterial") ==
        F.col("UrbanThread_NormalizedMaterial")
    )

    .withColumn(
        "DescriptionSimilarity",
        jaccard_similarity(
            "StyleHaus_NormalizedDescription",
            "UrbanThread_NormalizedDescription"
        )
    )

    .withColumn(
        "PriceDifference",
        F.abs(
            F.col("StyleHaus_Price") -
            F.col("UrbanThread_Price")
        ).cast(T.DecimalType(10, 2))
    )

    .withColumn(
        "PriceDifferencePct",
        F.when(
            F.greatest(
                F.abs(F.col("StyleHaus_Price")),
                F.abs(F.col("UrbanThread_Price"))
            ) == 0,
            F.lit(0.0)
        ).otherwise(
            (
                F.abs(
                    F.col("StyleHaus_Price") -
                    F.col("UrbanThread_Price")
                )
                /
                F.greatest(
                    F.abs(F.col("StyleHaus_Price")),
                    F.abs(F.col("UrbanThread_Price"))
                )
            ) * 100
        )
    )

    .withColumn(
        "BlockingRule",
        F.lit("CANONICAL_BRAND")
    )

    .withColumn(
        "CandidateGenerationTimestamp",
        F.current_timestamp()
    )
)

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 10, Finished, Available, Finished, False)

In [9]:
# ==============================================================================
# CELL 8 – APPLY FINAL 54-COLUMN CANDIDATE PAIR SCHEMA
# ==============================================================================

candidate_output_columns = [
    "CandidatePairID",
    "CandidateSchemaVersion",
    "StyleHaus_ProductID",
    "UrbanThread_ProductID",
    "StyleHaus_BatchID",
    "UrbanThread_BatchID",
    "StyleHaus_IngestionTimestamp",
    "UrbanThread_IngestionTimestamp",
    "StyleHaus_ProductName",
    "UrbanThread_ProductName",
    "StyleHaus_Brand",
    "UrbanThread_Brand",
    "StyleHaus_Category",
    "UrbanThread_Category",
    "StyleHaus_Description",
    "UrbanThread_Description",
    "StyleHaus_Price",
    "UrbanThread_Price",
    "StyleHaus_Color",
    "UrbanThread_Color",
    "StyleHaus_Size",
    "UrbanThread_Size",
    "StyleHaus_Material",
    "UrbanThread_Material",
    "StyleHaus_InStock",
    "UrbanThread_InStock",
    "StyleHaus_UpdatedAt",
    "UrbanThread_UpdatedAt",
    "CanonicalBrand",
    "StyleHaus_NormalizedProductName",
    "UrbanThread_NormalizedProductName",
    "StyleHaus_NormalizedCategory",
    "UrbanThread_NormalizedCategory",
    "StyleHaus_NormalizedColor",
    "UrbanThread_NormalizedColor",
    "StyleHaus_NormalizedSize",
    "UrbanThread_NormalizedSize",
    "StyleHaus_NormalizedMaterial",
    "UrbanThread_NormalizedMaterial",
    "StyleHaus_NormalizedDescription",
    "UrbanThread_NormalizedDescription",
    "CanonicalBrandMatch",
    "OriginalBrandExactMatch",
    "ProductNameExactMatch",
    "ProductNameSimilarity",
    "CategoryMatch",
    "ColorMatch",
    "SizeMatch",
    "MaterialMatch",
    "DescriptionSimilarity",
    "PriceDifference",
    "PriceDifferencePct",
    "BlockingRule",
    "CandidateGenerationTimestamp"
]

candidate_pair_schema_df = (
    candidate_pairs_with_signals_df
    .select(*candidate_output_columns)
)

total_candidate_pairs = candidate_pair_schema_df.count()
total_candidate_columns = len(candidate_pair_schema_df.columns)

print("Final candidate pairs:", total_candidate_pairs)
print("Final candidate columns:", total_candidate_columns)

assert total_candidate_pairs == EXPECTED_CANDIDATE_COUNT
assert total_candidate_columns == EXPECTED_OUTPUT_COLUMNS

display(candidate_pair_schema_df.limit(10))

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 11, Finished, Available, Finished, False)

Final candidate pairs: 185
Final candidate columns: 54


SynapseWidget(Synapse.DataFrame, edd17f19-3dea-4653-94fe-9c91c30f071e)

In [10]:
# ==============================================================================
# CELL 9 –: STRUCTURAL INTEGRITY VALIDATION
# ==============================================================================

unique_candidate_ids = (
    candidate_pair_schema_df
    .select("CandidatePairID")
    .distinct()
    .count()
)

null_candidate_ids = (
    candidate_pair_schema_df
    .filter(F.col("CandidatePairID").isNull())
    .count()
)

duplicate_candidate_ids = total_candidate_pairs - unique_candidate_ids

duplicate_product_pairs = (
    candidate_pair_schema_df
    .groupBy("StyleHaus_ProductID", "UrbanThread_ProductID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

structural_invalid_counts = {
    "NullCandidatePairIDs": null_candidate_ids,
    "DuplicateCandidatePairIDs": duplicate_candidate_ids,
    "NullStyleHausProductIDs": candidate_pair_schema_df.filter(
        F.col("StyleHaus_ProductID").isNull()
    ).count(),
    "NullUrbanThreadProductIDs": candidate_pair_schema_df.filter(
        F.col("UrbanThread_ProductID").isNull()
    ).count(),
    "DuplicateProductPairs": duplicate_product_pairs,
    "NullCanonicalBrands": candidate_pair_schema_df.filter(
        F.col("CanonicalBrand").isNull() |
        (F.trim(F.col("CanonicalBrand")) == "")
    ).count(),
    "NullBlockingRules": candidate_pair_schema_df.filter(
        F.col("BlockingRule").isNull()
    ).count(),
    "NullGenerationTimestamps": candidate_pair_schema_df.filter(
        F.col("CandidateGenerationTimestamp").isNull()
    ).count()
}

print("Structural validation:")
for name, value in structural_invalid_counts.items():
    print(name, "=", value)

assert unique_candidate_ids == total_candidate_pairs
assert all(v == 0 for v in structural_invalid_counts.values())
print("PASS: Structural Integrity")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 12, Finished, Available, Finished, False)

Structural validation:
NullCandidatePairIDs = 0
DuplicateCandidatePairIDs = 0
NullStyleHausProductIDs = 0
NullUrbanThreadProductIDs = 0
DuplicateProductPairs = 0
NullCanonicalBrands = 0
NullBlockingRules = 0
NullGenerationTimestamps = 0
PASS: Step 4.5.1 Structural Integrity


In [11]:
# ==============================================================================
# CELL 10 –SOURCE REFERENTIAL INTEGRITY
# ==============================================================================

stylehaus_reference_df = (
    silver_df
    .filter(F.col("SourceSystem") == STYLEHAUS_SOURCE)
    .select(
        F.col("SourceProductID").alias("StyleHaus_ProductID")
    )
    .distinct()
)

urbanthread_reference_df = (
    silver_df
    .filter(F.col("SourceSystem") == URBANTHREAD_SOURCE)
    .select(
        F.col("SourceProductID").alias("UrbanThread_ProductID")
    )
    .distinct()
)

invalid_stylehaus_ids_df = (
    candidate_pair_schema_df
    .select("CandidatePairID", "StyleHaus_ProductID")
    .join(
        stylehaus_reference_df,
        on="StyleHaus_ProductID",
        how="left_anti"
    )
)

invalid_urbanthread_ids_df = (
    candidate_pair_schema_df
    .select("CandidatePairID", "UrbanThread_ProductID")
    .join(
        urbanthread_reference_df,
        on="UrbanThread_ProductID",
        how="left_anti"
    )
)

invalid_stylehaus_count = invalid_stylehaus_ids_df.count()
invalid_urbanthread_count = invalid_urbanthread_ids_df.count()

wrong_stylehaus_source_count = (
    candidate_pair_schema_df.alias("CP")
    .join(
        silver_df.alias("S"),
        F.col("CP.StyleHaus_ProductID") == F.col("S.SourceProductID"),
        "inner"
    )
    .filter(F.col("S.SourceSystem") != STYLEHAUS_SOURCE)
    .select("CP.CandidatePairID")
    .distinct()
    .count()
)

wrong_urbanthread_source_count = (
    candidate_pair_schema_df.alias("CP")
    .join(
        silver_df.alias("S"),
        F.col("CP.UrbanThread_ProductID") == F.col("S.SourceProductID"),
        "inner"
    )
    .filter(F.col("S.SourceSystem") != URBANTHREAD_SOURCE)
    .select("CP.CandidatePairID")
    .distinct()
    .count()
)

fully_valid_candidate_count = (
    candidate_pair_schema_df
    .join(stylehaus_reference_df, "StyleHaus_ProductID", "inner")
    .join(urbanthread_reference_df, "UrbanThread_ProductID", "inner")
    .count()
)

unresolved_candidate_count = (
    total_candidate_pairs - fully_valid_candidate_count
)

print("Invalid StyleHaus IDs:", invalid_stylehaus_count)
print("Invalid UrbanThread IDs:", invalid_urbanthread_count)
print("Wrong StyleHaus source ownership:", wrong_stylehaus_source_count)
print("Wrong UrbanThread source ownership:", wrong_urbanthread_source_count)
print("Fully valid candidate references:", fully_valid_candidate_count)
print("Unresolved references:", unresolved_candidate_count)

assert invalid_stylehaus_count == 0
assert invalid_urbanthread_count == 0
assert wrong_stylehaus_source_count == 0
assert wrong_urbanthread_source_count == 0
assert fully_valid_candidate_count == total_candidate_pairs
assert unresolved_candidate_count == 0

print("PASS: Source Referential Integrity")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 13, Finished, Available, Finished, False)

Invalid StyleHaus IDs: 0
Invalid UrbanThread IDs: 0
Wrong StyleHaus source ownership: 0
Wrong UrbanThread source ownership: 0
Fully valid candidate references: 185
Unresolved references: 0
PASS: Step 4.5.2 Source Referential Integrity


In [12]:
# ==============================================================================
# CELL 11 –SOURCE LINEAGE VALIDATION
# ==============================================================================

stylehaus_source_lineage_df = (
    silver_df
    .filter(F.col("SourceSystem") == STYLEHAUS_SOURCE)
    .select(
        F.col("SourceProductID").alias("StyleHaus_ProductID"),
        F.col("BatchID").alias("Expected_StyleHaus_BatchID"),
        F.col("IngestionTimestamp").alias(
            "Expected_StyleHaus_IngestionTimestamp"
        )
    )
)

urbanthread_source_lineage_df = (
    silver_df
    .filter(F.col("SourceSystem") == URBANTHREAD_SOURCE)
    .select(
        F.col("SourceProductID").alias("UrbanThread_ProductID"),
        F.col("BatchID").alias("Expected_UrbanThread_BatchID"),
        F.col("IngestionTimestamp").alias(
            "Expected_UrbanThread_IngestionTimestamp"
        )
    )
)

lineage_comparison_df = (
    candidate_pair_schema_df.alias("CP")
    .join(
        stylehaus_source_lineage_df.alias("SH"),
        F.col("CP.StyleHaus_ProductID") == F.col("SH.StyleHaus_ProductID"),
        "inner"
    )
    .join(
        urbanthread_source_lineage_df.alias("UT"),
        F.col("CP.UrbanThread_ProductID") == F.col("UT.UrbanThread_ProductID"),
        "inner"
    )
    .select(
        F.col("CP.CandidatePairID"),
        F.col("CP.StyleHaus_BatchID"),
        F.col("SH.Expected_StyleHaus_BatchID"),
        F.col("CP.UrbanThread_BatchID"),
        F.col("UT.Expected_UrbanThread_BatchID"),
        F.col("CP.StyleHaus_IngestionTimestamp"),
        F.col("SH.Expected_StyleHaus_IngestionTimestamp"),
        F.col("CP.UrbanThread_IngestionTimestamp"),
        F.col("UT.Expected_UrbanThread_IngestionTimestamp")
    )
)

lineage_mismatch_df = lineage_comparison_df.filter(
    (~F.col("StyleHaus_BatchID").eqNullSafe(
        F.col("Expected_StyleHaus_BatchID")
    ))
    |
    (~F.col("UrbanThread_BatchID").eqNullSafe(
        F.col("Expected_UrbanThread_BatchID")
    ))
    |
    (~F.col("StyleHaus_IngestionTimestamp").eqNullSafe(
        F.col("Expected_StyleHaus_IngestionTimestamp")
    ))
    |
    (~F.col("UrbanThread_IngestionTimestamp").eqNullSafe(
        F.col("Expected_UrbanThread_IngestionTimestamp")
    ))
)

lineage_comparison_count = lineage_comparison_df.count()
lineage_mismatch_count = lineage_mismatch_df.count()

print("Lineage comparison records:", lineage_comparison_count)
print("Lineage mismatches:", lineage_mismatch_count)

assert lineage_comparison_count == total_candidate_pairs
assert lineage_mismatch_count == 0

print("PASS: Source Lineage")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 14, Finished, Available, Finished, False)

Lineage comparison records: 185
Lineage mismatches: 0
PASS: Step 4.5.3 Source Lineage


In [13]:
# ==============================================================================
# CELL 12 –  MATCHING SIGNAL VALIDATION
# ==============================================================================

# Recalculate expected values independently from the stored signal columns.
matching_signal_recalculated_df = (
    candidate_pair_schema_df

    .withColumn(
        "Expected_CanonicalBrandMatch",
        F.col("CanonicalBrand").isNotNull() &
        (F.trim(F.col("CanonicalBrand")) != "") &
        (F.col("BlockingRule") == "CANONICAL_BRAND")
    )

    .withColumn(
        "Expected_OriginalBrandExactMatch",
        F.lower(F.trim(F.col("StyleHaus_Brand"))) ==
        F.lower(F.trim(F.col("UrbanThread_Brand")))
    )

    .withColumn(
        "Expected_ProductNameExactMatch",
        F.col("StyleHaus_NormalizedProductName") ==
        F.col("UrbanThread_NormalizedProductName")
    )

    .withColumn(
        "Expected_ProductNameSimilarity",
        jaccard_similarity(
            "StyleHaus_NormalizedProductName",
            "UrbanThread_NormalizedProductName"
        )
    )

    .withColumn(
        "Expected_CategoryMatch",
        F.col("StyleHaus_NormalizedCategory") ==
        F.col("UrbanThread_NormalizedCategory")
    )

    .withColumn(
        "Expected_ColorMatch",
        F.col("StyleHaus_NormalizedColor") ==
        F.col("UrbanThread_NormalizedColor")
    )

    .withColumn(
        "Expected_SizeMatch",
        F.col("StyleHaus_NormalizedSize") ==
        F.col("UrbanThread_NormalizedSize")
    )

    .withColumn(
        "Expected_MaterialMatch",
        F.col("StyleHaus_NormalizedMaterial") ==
        F.col("UrbanThread_NormalizedMaterial")
    )

    .withColumn(
        "Expected_DescriptionSimilarity",
        jaccard_similarity(
            "StyleHaus_NormalizedDescription",
            "UrbanThread_NormalizedDescription"
        )
    )

    .withColumn(
        "Expected_PriceDifference",
        F.abs(
            F.col("StyleHaus_Price") -
            F.col("UrbanThread_Price")
        )
    )

    .withColumn(
        "Expected_PriceDifferencePct",
        F.when(
            F.greatest(
                F.abs(F.col("StyleHaus_Price")),
                F.abs(F.col("UrbanThread_Price"))
            ) == 0,
            F.lit(0.0)
        ).otherwise(
            (
                F.abs(
                    F.col("StyleHaus_Price") -
                    F.col("UrbanThread_Price")
                )
                /
                F.greatest(
                    F.abs(F.col("StyleHaus_Price")),
                    F.abs(F.col("UrbanThread_Price"))
                )
            ) * 100
        )
    )
)

boolean_signal_pairs = [
    ("CanonicalBrandMatch", "Expected_CanonicalBrandMatch"),
    ("OriginalBrandExactMatch", "Expected_OriginalBrandExactMatch"),
    ("ProductNameExactMatch", "Expected_ProductNameExactMatch"),
    ("CategoryMatch", "Expected_CategoryMatch"),
    ("ColorMatch", "Expected_ColorMatch"),
    ("SizeMatch", "Expected_SizeMatch"),
    ("MaterialMatch", "Expected_MaterialMatch")
]

comparison_df = matching_signal_recalculated_df

for actual_col, expected_col in boolean_signal_pairs:
    comparison_df = comparison_df.withColumn(
        f"Mismatch_{actual_col}",
        ~F.col(actual_col).eqNullSafe(F.col(expected_col))
    )

numeric_signal_pairs = [
    ("ProductNameSimilarity", "Expected_ProductNameSimilarity"),
    ("DescriptionSimilarity", "Expected_DescriptionSimilarity"),
    ("PriceDifference", "Expected_PriceDifference"),
    ("PriceDifferencePct", "Expected_PriceDifferencePct")
]

for actual_col, expected_col in numeric_signal_pairs:
    comparison_df = comparison_df.withColumn(
        f"Mismatch_{actual_col}",
        F.col(actual_col).isNull() |
        F.col(expected_col).isNull() |
        (
            F.abs(
                F.col(actual_col).cast("double") -
                F.col(expected_col).cast("double")
            ) > SIMILARITY_TOLERANCE
        )
    )

signal_mismatch_columns = [
    f"Mismatch_{actual_col}"
    for actual_col, _ in boolean_signal_pairs + numeric_signal_pairs
]

signal_validation_results = []

for mismatch_column in signal_mismatch_columns:
    mismatch_count = (
        comparison_df
        .filter(F.col(mismatch_column) == True)
        .count()
    )

    signal_validation_results.append(
        (
            mismatch_column.replace("Mismatch_", ""),
            mismatch_count,
            "PASS" if mismatch_count == 0 else "FAIL"
        )
    )

signal_validation_summary_df = spark.createDataFrame(
    signal_validation_results,
    ["MatchingSignal", "MismatchCount", "ValidationStatus"]
)

display(signal_validation_summary_df)

total_matching_signals = len(signal_mismatch_columns)
total_signal_checks = total_candidate_pairs * total_matching_signals
total_signal_mismatches = sum(
    row[1] for row in signal_validation_results
)

mismatch_condition = F.lit(False)
for mismatch_column in signal_mismatch_columns:
    mismatch_condition = (
        mismatch_condition | (F.col(mismatch_column) == True)
    )

matching_signal_mismatches_df = comparison_df.filter(mismatch_condition)
matching_signal_mismatch_count = matching_signal_mismatches_df.count()

print("Total matching signals:", total_matching_signals)
print("Total signal checks:", total_signal_checks)
print("Total signal mismatches:", total_signal_mismatches)
print("Candidate pairs with signal mismatches:", matching_signal_mismatch_count)

assert total_signal_mismatches == 0
assert matching_signal_mismatch_count == 0

print("PASS: Matching Signal Validation")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9df50b19-066d-4125-bd80-f49c991b80c1)

Total matching signals: 11
Total signal checks: 2035
Total signal mismatches: 0
Candidate pairs with signal mismatches: 0
PASS: Step 4.5.4 Matching Signal Validation


In [14]:
# ==============================================================================
# CELL 13 – CANDIDATE COVERAGE VALIDATION
# ==============================================================================

brand_source_counts_df = (
    candidate_base_df
    .groupBy("CanonicalBrand", "SourceSystem")
    .agg(
        F.countDistinct("SourceProductID").alias("ProductCount")
    )
)

brand_coverage_df = (
    brand_source_counts_df
    .groupBy("CanonicalBrand")
    .pivot(
        "SourceSystem",
        [STYLEHAUS_SOURCE, URBANTHREAD_SOURCE]
    )
    .agg(F.first("ProductCount"))
    .fillna(0)
)

overlapping_brands_df = (
    brand_coverage_df
    .filter(
        (F.col(STYLEHAUS_SOURCE) > 0) &
        (F.col(URBANTHREAD_SOURCE) > 0)
    )
)

expected_brand_candidates_df = (
    overlapping_brands_df
    .withColumn(
        "ExpectedCandidatePairs",
        F.col(STYLEHAUS_SOURCE) * F.col(URBANTHREAD_SOURCE)
    )
    .select(
        "CanonicalBrand",
        F.col(STYLEHAUS_SOURCE).alias("StyleHausProducts"),
        F.col(URBANTHREAD_SOURCE).alias("UrbanThreadProducts"),
        "ExpectedCandidatePairs"
    )
)

actual_brand_candidates_df = (
    candidate_pair_schema_df
    .groupBy("CanonicalBrand")
    .agg(
        F.count("*").alias("ActualCandidatePairs")
    )
)

coverage_comparison_df = (
    expected_brand_candidates_df.alias("E")
    .join(
        actual_brand_candidates_df.alias("A"),
        on="CanonicalBrand",
        how="full_outer"
    )
    .select(
        "CanonicalBrand",
        F.coalesce(F.col("StyleHausProducts"), F.lit(0)).alias(
            "StyleHausProducts"
        ),
        F.coalesce(F.col("UrbanThreadProducts"), F.lit(0)).alias(
            "UrbanThreadProducts"
        ),
        F.coalesce(F.col("ExpectedCandidatePairs"), F.lit(0)).alias(
            "ExpectedCandidatePairs"
        ),
        F.coalesce(F.col("ActualCandidatePairs"), F.lit(0)).alias(
            "ActualCandidatePairs"
        )
    )
    .withColumn(
        "CandidateDifference",
        F.col("ExpectedCandidatePairs") -
        F.col("ActualCandidatePairs")
    )
    .withColumn(
        "CoverageStatus",
        F.when(F.col("CandidateDifference") == 0, "PASS")
         .otherwise("FAIL")
    )
)

display(coverage_comparison_df.orderBy("CanonicalBrand"))

coverage_summary = (
    coverage_comparison_df
    .agg(
        F.sum("ExpectedCandidatePairs").alias("ExpectedPairs"),
        F.sum("ActualCandidatePairs").alias("ActualPairs"),
        F.sum(F.abs(F.col("CandidateDifference"))).alias(
            "TotalAbsoluteDifference"
        )
    )
    .first()
)

total_overlapping_brands = overlapping_brands_df.count()
total_expected_pairs = coverage_summary["ExpectedPairs"]
total_actual_pairs = coverage_summary["ActualPairs"]
total_coverage_difference = coverage_summary["TotalAbsoluteDifference"]

total_brand_mismatches = (
    coverage_comparison_df
    .filter(F.col("CoverageStatus") == "FAIL")
    .count()
)

print("Overlapping canonical brands:", total_overlapping_brands)
print("Expected candidate pairs:", total_expected_pairs)
print("Actual candidate pairs:", total_actual_pairs)
print("Brand-level coverage mismatches:", total_brand_mismatches)
print("Total coverage difference:", total_coverage_difference)

assert total_expected_pairs == total_actual_pairs
assert total_coverage_difference == 0
assert total_brand_mismatches == 0

print("PASS: Candidate Coverage")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2cbf46c-330f-44f1-933d-1b716fe84f2a)

Overlapping canonical brands: 35
Expected candidate pairs: 185
Actual candidate pairs: 185
Brand-level coverage mismatches: 0
Total coverage difference: 0
PASS: Step 4.5.5 Candidate Coverage


In [15]:
# ==============================================================================
# CELL 14 – ANDIDATE GENERATION EFFICIENCY
# ==============================================================================

total_possible_comparisons = (
    stylehaus_count * urbanthread_count
)

actual_candidate_pairs = total_candidate_pairs

eliminated_comparisons = (
    total_possible_comparisons - actual_candidate_pairs
)

comparison_reduction_pct = (
    eliminated_comparisons / total_possible_comparisons * 100
    if total_possible_comparisons else 0.0
)

candidate_retention_pct = (
    actual_candidate_pairs / total_possible_comparisons * 100
    if total_possible_comparisons else 0.0
)

print("Total possible comparisons:", total_possible_comparisons)
print("Generated candidate pairs:", actual_candidate_pairs)
print("Eliminated comparisons:", eliminated_comparisons)
print("Comparison reduction (%):", round(comparison_reduction_pct, 2))
print("Candidate retention (%):", round(candidate_retention_pct, 2))

assert total_possible_comparisons == 6580
assert actual_candidate_pairs == 185
assert eliminated_comparisons == 6395
assert round(comparison_reduction_pct, 2) == 97.19
assert round(candidate_retention_pct, 2) == 2.81

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 17, Finished, Available, Finished, False)

Total possible comparisons: 6580
Generated candidate pairs: 185
Eliminated comparisons: 6395
Comparison reduction (%): 97.19
Candidate retention (%): 2.81


In [16]:
# ==============================================================================
# CELL 15 – FINAL VALIDATION SUMMARY
# ==============================================================================

validation_summary_results = []


def add_validation_result(area, name, expected, actual):
    status = "PASS" if expected == actual else "FAIL"
    validation_summary_results.append(
        (area, name, str(expected), str(actual), status)
    )


# 4.5.1 – Structural integrity: 11 checks
add_validation_result(
    "Structural Integrity",
    "Total Candidate Records",
    EXPECTED_CANDIDATE_COUNT,
    total_candidate_pairs
)
add_validation_result(
    "Structural Integrity",
    "Total Columns",
    EXPECTED_OUTPUT_COLUMNS,
    total_candidate_columns
)
add_validation_result(
    "Structural Integrity",
    "Unique Candidate IDs",
    total_candidate_pairs,
    unique_candidate_ids
)
add_validation_result(
    "Structural Integrity",
    "Null Candidate IDs",
    0,
    null_candidate_ids
)
add_validation_result(
    "Structural Integrity",
    "Duplicate Candidate IDs",
    0,
    duplicate_candidate_ids
)
add_validation_result(
    "Structural Integrity",
    "Duplicate Product Pairs",
    0,
    duplicate_product_pairs
)
add_validation_result(
    "Structural Integrity",
    "Null StyleHaus Product IDs",
    0,
    structural_invalid_counts["NullStyleHausProductIDs"]
)
add_validation_result(
    "Structural Integrity",
    "Null UrbanThread Product IDs",
    0,
    structural_invalid_counts["NullUrbanThreadProductIDs"]
)
add_validation_result(
    "Structural Integrity",
    "Null Canonical Brands",
    0,
    structural_invalid_counts["NullCanonicalBrands"]
)
add_validation_result(
    "Structural Integrity",
    "Null Blocking Rules",
    0,
    structural_invalid_counts["NullBlockingRules"]
)
add_validation_result(
    "Structural Integrity",
    "Null Generation Timestamps",
    0,
    structural_invalid_counts["NullGenerationTimestamps"]
)

# 4.5.2 – Referential integrity: 5 checks
add_validation_result(
    "Referential Integrity",
    "Invalid StyleHaus Product IDs",
    0,
    invalid_stylehaus_count
)
add_validation_result(
    "Referential Integrity",
    "Invalid UrbanThread Product IDs",
    0,
    invalid_urbanthread_count
)
add_validation_result(
    "Referential Integrity",
    "Wrong StyleHaus Source Ownership",
    0,
    wrong_stylehaus_source_count
)
add_validation_result(
    "Referential Integrity",
    "Wrong UrbanThread Source Ownership",
    0,
    wrong_urbanthread_source_count
)
add_validation_result(
    "Referential Integrity",
    "Fully Valid Candidate References",
    total_candidate_pairs,
    fully_valid_candidate_count
)

# 4.5.3 – Source lineage: 2 checks
add_validation_result(
    "Source Lineage",
    "Lineage Comparison Records",
    total_candidate_pairs,
    lineage_comparison_count
)
add_validation_result(
    "Source Lineage",
    "Lineage Mismatches",
    0,
    lineage_mismatch_count
)

# 4.5.4 – Matching signals: 3 checks
add_validation_result(
    "Matching Signal",
    "Total Matching Signals",
    11,
    total_matching_signals
)
add_validation_result(
    "Matching Signal",
    "Total Signal Checks",
    total_candidate_pairs * 11,
    total_signal_checks
)
add_validation_result(
    "Matching Signal",
    "Total Signal Mismatches",
    0,
    total_signal_mismatches
)

# 4.5.5 – Candidate coverage: 5 checks
add_validation_result(
    "Candidate Coverage",
    "Overlapping Canonical Brands",
    35,
    total_overlapping_brands
)
add_validation_result(
    "Candidate Coverage",
    "Expected Candidate Pairs",
    EXPECTED_CANDIDATE_COUNT,
    total_expected_pairs
)
add_validation_result(
    "Candidate Coverage",
    "Actual Candidate Pairs",
    EXPECTED_CANDIDATE_COUNT,
    total_actual_pairs
)
add_validation_result(
    "Candidate Coverage",
    "Brand-Level Coverage Mismatches",
    0,
    total_brand_mismatches
)
add_validation_result(
    "Candidate Coverage",
    "Total Coverage Difference",
    0,
    total_coverage_difference
)

# Generation efficiency: 4 checks
add_validation_result(
    "Generation Efficiency",
    "Total Possible Comparisons",
    6580,
    total_possible_comparisons
)
add_validation_result(
    "Generation Efficiency",
    "Eliminated Comparisons",
    6395,
    eliminated_comparisons
)
add_validation_result(
    "Generation Efficiency",
    "Comparison Reduction (%)",
    97.19,
    round(comparison_reduction_pct, 2)
)
add_validation_result(
    "Generation Efficiency",
    "Candidate Retention (%)",
    2.81,
    round(candidate_retention_pct, 2)
)

final_validation_summary_df = spark.createDataFrame(
    validation_summary_results,
    [
        "ValidationArea",
        "ValidationName",
        "ExpectedValue",
        "ActualValue",
        "ValidationStatus"
    ]
)

display(
    final_validation_summary_df
    .orderBy("ValidationArea", "ValidationName")
)

total_validation_checks = len(validation_summary_results)

passed_checks = sum(
    1 for row in validation_summary_results
    if row[4] == "PASS"
)

failed_checks = total_validation_checks - passed_checks

print("======================================================")
print("PHASE 4.5 – FINAL VALIDATION SUMMARY")
print("======================================================")
print("Total Candidate Pairs:", total_candidate_pairs)
print("Total Validation Checks:", total_validation_checks)
print("Passed Checks:", passed_checks)
print("Failed Checks:", failed_checks)
print("Comparison Reduction (%):", round(comparison_reduction_pct, 2))
print("Candidate Retention (%):", round(candidate_retention_pct, 2))

assert total_validation_checks == 30
assert passed_checks == 30
assert failed_checks == 0

print("PASS: ALL VALIDATIONS COMPLETED")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0454655f-4653-46f4-9a13-b755f9dc556d)

PHASE 4.5 – FINAL VALIDATION SUMMARY
Total Candidate Pairs: 185
Total Validation Checks: 30
Passed Checks: 30
Failed Checks: 0
Comparison Reduction (%): 97.19
Candidate Retention (%): 2.81
PASS: PHASE 4.5 – ALL VALIDATIONS COMPLETED


In [17]:
# ==============================================================================
# CELL 16 – PERSIST FINAL CANDIDATE PAIRS TO THE LAKEHOUSE
# ==============================================================================
# Persist only after all Phase 4 validations pass.
# This creates/replaces the managed Delta table ProductCandidatePairs.
# ==============================================================================

(
    candidate_pair_schema_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print(f"Table {OUTPUT_TABLE} written successfully.")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 19, Finished, Available, Finished, False)

Table semantic.ProductCandidatePairs written successfully.


In [18]:
# ==============================================================================
# CELL 17 – READ-BACK VALIDATION
# ==============================================================================

persisted_candidate_df = spark.read.table(OUTPUT_TABLE)

persisted_count = persisted_candidate_df.count()
persisted_column_count = len(persisted_candidate_df.columns)

print("Persisted table:", OUTPUT_TABLE)
print("Persisted records:", persisted_count)
print("Persisted columns:", persisted_column_count)

assert persisted_count == EXPECTED_CANDIDATE_COUNT
assert persisted_column_count == EXPECTED_OUTPUT_COLUMNS

persisted_duplicate_ids = (
    persisted_candidate_df
    .groupBy("CandidatePairID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert persisted_duplicate_ids == 0

print("PASS: ProductCandidatePairs persisted and read-back validated.")
display(persisted_candidate_df.limit(10))

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 20, Finished, Available, Finished, False)

Persisted table: semantic.ProductCandidatePairs
Persisted records: 185
Persisted columns: 54
PASS: ProductCandidatePairs persisted and read-back validated.


SynapseWidget(Synapse.DataFrame, 21e8e49d-c5b7-45d0-9d0a-aab65661140c)

In [19]:
# ==============================================================================
# PHASE 4 – COMPLETION
# ==============================================================================

print("======================================================")
print("PHASE 4 – CANDIDATE GENERATION COMPLETED")
print("======================================================")
print("Input table:", SOURCE_TABLE)
print("Output table:", OUTPUT_TABLE)
print("Candidate pairs:", persisted_count)
print("Candidate schema columns:", persisted_column_count)
print("Validation checks passed:", passed_checks)
print("Validation checks failed:", failed_checks)
print("Comparison reduction (%):", round(comparison_reduction_pct, 2))
print("======================================================")
print("READY FOR PHASE 5 – AI-BASED PRODUCT MATCHING")
print("======================================================")

StatementMeta(, e1f67a5a-ff74-4e38-bd50-677be4246a51, 21, Finished, Available, Finished, False)

PHASE 4 – CANDIDATE GENERATION COMPLETED
Input table: Silver.Products
Output table: semantic.ProductCandidatePairs
Candidate pairs: 185
Candidate schema columns: 54
Validation checks passed: 30
Validation checks failed: 0
Comparison reduction (%): 97.19
READY FOR PHASE 5 – AI-BASED PRODUCT MATCHING
